# LSTM Alphabet Sequence Prediction

# Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.utils import to_categorical
from keras.optimizers import Adam

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Print version information
import keras
print("Keras version:", keras.__version__)

# Create Mappings and Dataset
Dataset is just "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

Map characters to (0-25) and vice versa

Generate Dataset (Input: A letter, Output: Next letter of the alphabet)

Reshape features X to be [samples, time steps, features]

In [ ]:
# Define the dataset
dataset = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# Create character to integer mapping and vice versa
char_to_int = {c: i for i, c in enumerate(dataset)}
int_to_char = {i: c for i, c in enumerate(dataset)}

# Sanity check: Print the character mappings
print("Character to Integer Mapping:", char_to_int)
print("Integer to Character Mapping:", int_to_char)

In [ ]:
# Generate the dataset
def generate_dataset(length=1000):
    X, y = [], []
    for _ in range(length):
        tmp = np.random.randint(0, 25)
        X.append(tmp)
        y.append(tmp + 1 if tmp < 25 else 0)
    return np.array(X), np.array(y)

# Generate the training dataset
X, y = generate_dataset()

# Generate the test dataset
X_test, y_test = generate_dataset()

# Sanity check: Print the first 10 samples
print("First 10 samples of X:", X[:10])
print("First 10 samples of y:", y[:10])

In [ ]:
# Reshape features X to be [samples, time steps, features]
X = X.reshape((X.shape[0], 1, 1))
X_test = X_test.reshape((X_test.shape[0], 1, 1))

# Sanity check: Print the shape of X
print("Shape of X after reshaping:", X.shape)

## Normalize the Dataset

In [ ]:
# Normalize the input
X = X / float(len(dataset))
X_test = X_test / float(len(dataset))

# Sanity check: Print the first 10 samples of X
print("First 10 samples of X after reshaping:", X[:10])
print(X.shape)

## One Hot Encode Output

In [ ]:
# Convert output to categorical
y = to_categorical(y, num_classes=len(dataset))
y_test = to_categorical(y_test, num_classes=len(dataset))

print("First 10 samples of y after one-hot encoding:", y[:10])

# Create Model

In [ ]:
# Create the LSTM model
def create_model(num_layers, num_units, lr=0.001):

    model = Sequential()

    # Add LSTM layers
    for i in range(num_layers):
        if i == 0:
            # Need to make sure return_sequences is True for all but the last layer
            model.add(LSTM(num_units[i], input_shape=(1, 1), return_sequences=(num_layers > 1)))
        else:
            model.add(LSTM(num_units[i], return_sequences=(i < num_layers - 1)))

    model.add(Dense(len(dataset), activation='softmax'))

    # Set optimizer with a given learning rate
    optimizer = Adam(learning_rate=lr)

    # Compile the model
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

# Create the first model
model = create_model(num_layers=1, num_units=[4])

## Fit Model

In [ ]:
# Train the model
model.fit(X, y, epochs=100, batch_size=1, verbose=2)

# Evaluate
Print accuracy

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

# Demonstrate Model Predictions
Run the model against 20 points and print expected character and actual character.

In [ ]:
# Generate examples
X_example, y_example = generate_dataset(length=20)
X_example = X_example.reshape((X_example.shape[0], 1, 1))
X_example_normalized = X_example / float(len(dataset))
y_example = to_categorical(y_example, num_classes=len(dataset))

# Demonstrate Model Predictions (Print input, expected, and predicted characters)
for i in range(len(X_example)):
    input_char = int_to_char[np.argmax(X_example[i])]
    expected_char = int_to_char[np.argmax(y_example[i])]
    predicted_char = int_to_char[np.argmax(model.predict(X_example_normalized[i]))]
    print(f"Input: {input_char}, Expected: {expected_char}, Predicted: {predicted_char}")

# Experiments

Two LSTM Layers vs More LSTM Layers (Plot and graph time + accuracy)

In [ ]:
# Initialize two layer LSTM model
model_2_layers = create_model(num_layers=2, num_units=[4]*2)

# Initialize 4, 6, and 8 layer LSTM models
model_8_layers = create_model(num_layers=8, num_units=[4]*8)
model_16_layers = create_model(num_layers=16, num_units=[4]*16)

In [ ]:
# Train the models
model_2_layers.fit(X, y, epochs=100, batch_size=1, verbose=2)
model_8_layers.fit(X, y, epochs=100, batch_size=1, verbose=2)
model_16_layers.fit(X, y, epochs=100, batch_size=1, verbose=2)

In [ ]:
# Store the model evaluation accuracies for plotting
layers_nums = [2, 8, 16]
accuracies = []

# Evaluate the models and store accuracies
accuracies.append(model_2_layers.evaluate(X_test, y_test)[1])
accuracies.append(model_8_layers.evaluate(X_test, y_test)[1])
accuracies.append(model_16_layers.evaluate(X_test, y_test)[1])

# Plot the accuracies
plt.figure(figsize=(10, 5))
plt.plot(layers_nums, accuracies, marker='o')
plt.title('Model Accuracy vs Number of Layers')
plt.xlabel('Number of LSTM Layers')
plt.ylabel('Accuracy')
plt.xticks(layers_nums)
plt.grid()
plt.show()

Vary learning rate, hidden size, and number of layers. Use accuracy as the comparative metric.

In [ ]:
# Set the different parameters for the models
learning_rates = [0.001, 0.01, 0.1]
num_units = [5, 25, 50]
num_layers = [1, 8, 16]

# Initialize models with different parameters
models = []
for lr in learning_rates:
    for units in num_units:
        for layers in num_layers:
            model = create_model(num_layers=layers, num_units=[units]*layers, lr=lr)
            models.append(model)

# Train the models with different parameters
for model in models:
    model.fit(X, y, epochs=100, batch_size=1, verbose=2)

# Store the model evaluation accuracies for plotting
accuracies = []

# Evaluate the models and store accuracies
for model in models:
    accuracies.append(model.evaluate(X_test, y_test)[1])

In [ ]:
# Turn learning rates, number of units, and number of layers into columns of a pandas dataframe for easier visualization
acc_df = pd.DataFrame({
    'Learning Rate': [model.optimizer.learning_rate.numpy() for model in models],
    'Number of Units': [model.layers[0].units for model in models],
    'Number of Layers': [len(model.layers) for model in models],
    'Accuracy': accuracies
})

print(acc_df)

for layers in [2, 9, 17]:
    tmp = acc_df[acc_df['Number of Layers'] == layers]
    plt.figure(figsize=(10, 6))
    for lr in learning_rates:
        tmp_lr = tmp[tmp['Learning Rate'] == lr]
        plt.plot(tmp_lr['Number of Units'], tmp_lr['Accuracy'], marker='o', label=f'LR: {lr}')
        plt.xlabel('Number of Units')
    plt.ylabel('Accuracy')
    plt.title(f'Accuracy vs Number of Units for {layers-1} LSTM Layers')
    plt.legend()
    plt.show()